# GameForge AI: Dedicated Denoising Autoencoder (DAE) Training Pipeline

## Multi-GPU Support (2 x Kaggle T4 GPUs)
This Kaggle notebook implements a mathematically rigorous **Denoising Autoencoder (DAE)** trained specifically on **50,000 unique RGBA sprites** from the `evilsocket/alucard-sprites` dataset using **TensorFlow `MirroredStrategy` across dual Kaggle T4 GPUs**.

### Training Formulation:
Noisy Input x_tilde = x + N(0, sigma^2) -> DAE(x_tilde) -> Clean Target x

* **Input**: Synthetic Gaussian noise (stddev = 0.15) added to clean 128 x 128 x 4 RGBA sprites.
* **Target**: Original uncorrupted clean sprite ground truth (x).
* **Objective**: Minimize MSE(DAE(x_tilde), x) to learn noise attenuation and pixel restoration.

In [ ]:
import os
import random
import hashlib
import json
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("Detected GPU Devices:", gpus)

# Enable Multi-GPU Mirrored Strategy for 2x Kaggle T4 GPUs
strategy = tf.distribute.MirroredStrategy()
print(f"Number of GPU replicas in sync: {strategy.num_replicas_in_sync}")

BASE_DIR = "/kaggle/working/DAE_50K"
DIRS = {
    "checkpoints": os.path.join(BASE_DIR, "checkpoints"),
    "logs": os.path.join(BASE_DIR, "logs"),
    "config": os.path.join(BASE_DIR, "config"),
    "models": os.path.join(BASE_DIR, "models")
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)


## 1. Load HuggingFace Dataset & Select 50K Subset

In [ ]:
!pip install -q datasets Pillow
from datasets import load_dataset

print("Loading 'evilsocket/alucard-sprites' dataset...")
raw_ds = load_dataset("evilsocket/alucard-sprites", split="train")
print(f"Total raw samples in dataset: {len(raw_ds)}")

# Select 50,000 unique samples
TARGET_POPULATION = 50000
random.seed(42)
indices = list(range(len(raw_ds)))
random.shuffle(indices)
selected_indices = indices[:TARGET_POPULATION]

dataset_50k = raw_ds.select(selected_indices)

# Split into Train (45k), Val (2.5k), Test (2.5k)
TRAIN_SIZE = 45000
VAL_SIZE = 2500
TEST_SIZE = 2500

train_data = dataset_50k.select(range(0, TRAIN_SIZE))
val_data = dataset_50k.select(range(TRAIN_SIZE, TRAIN_SIZE + VAL_SIZE))
test_data = dataset_50k.select(range(TRAIN_SIZE + VAL_SIZE, TARGET_POPULATION))

print(f"Train subset: {len(train_data)}")
print(f"Validation subset: {len(val_data)}")
print(f"Test subset: {len(test_data)}")


## 2. Multi-GPU Data Pipeline (`Noisy Input -> Clean Target`)

Batch size scaled dynamically based on `strategy.num_replicas_in_sync` for optimal 2 x T4 throughput.

In [ ]:
NOISE_SCALE = 0.15
# Scale batch size proportionally for multi-GPU throughput
GLOBAL_BATCH_SIZE = 64 * strategy.num_replicas_in_sync
print(f"Global Multi-GPU Batch Size: {GLOBAL_BATCH_SIZE}")

def dae_data_generator(hf_dataset, noise_scale=0.15):
    for item in hf_dataset:
        clean_img = item["image"].convert("RGBA")
        clean_np = np.array(clean_img, dtype=np.float32) / 255.0
        noise = np.random.normal(loc=0.0, scale=noise_scale, size=clean_np.shape).astype(np.float32)
        noisy_np = np.clip(clean_np + noise, 0.0, 1.0)
        yield (noisy_np, clean_np)

def create_dae_tf_dataset(hf_dataset, is_training=True):
    ds = tf.data.Dataset.from_generator(
        lambda: dae_data_generator(hf_dataset, noise_scale=NOISE_SCALE),
        output_signature=(
            tf.TensorSpec(shape=(128, 128, 4), dtype=tf.float32),
            tf.TensorSpec(shape=(128, 128, 4), dtype=tf.float32)
        )
    )
    if is_training:
        ds = ds.batch(GLOBAL_BATCH_SIZE).repeat().prefetch(tf.data.AUTOTUNE)
    else:
        ds = ds.batch(GLOBAL_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = create_dae_tf_dataset(train_data, is_training=True)
val_ds = create_dae_tf_dataset(val_data, is_training=True)
test_ds = create_dae_tf_dataset(test_data, is_training=False)

print("Multi-GPU Denoising dataset generators ready.")


## 3. Denoising Autoencoder (DAE) Architecture Scope under MirroredStrategy

In [ ]:
# Wrap model creation and compilation inside MirroredStrategy scope for 2x T4 GPUs
with strategy.scope():
    inputs = keras.Input(shape=(128, 128, 4), name="noisy_input")
    x = layers.Conv2D(32, 3, padding="same", strides=2)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    
    x = layers.Conv2D(64, 3, padding="same", strides=2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    
    x = layers.Conv2D(128, 3, padding="same", strides=2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    
    x = layers.Conv2D(256, 3, padding="same", strides=2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    
    x = layers.Flatten()(x)
    bottleneck = layers.Dense(256, name="dae_bottleneck")(x)
    
    x = layers.Dense(8 * 8 * 256, activation="relu")(bottleneck)
    x = layers.Reshape((8, 8, 256))(x)
    
    x = layers.Conv2DTranspose(128, 4, padding="same", strides=2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    
    x = layers.Conv2DTranspose(64, 4, padding="same", strides=2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    
    x = layers.Conv2DTranspose(32, 4, padding="same", strides=2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    
    outputs = layers.Conv2DTranspose(4, 4, activation="sigmoid", padding="same", strides=2, name="denoised_output")(x)
    
    dae_model = keras.Model(inputs, outputs, name="Denoising_Autoencoder_50K")
    dae_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"]
    )

dae_model.summary()


## 4. Multi-GPU Distributed Training Pipeline (50K Dataset, 30 Epochs)

In [ ]:
EPOCHS = 30
STEPS_PER_EPOCH = TRAIN_SIZE // GLOBAL_BATCH_SIZE
VAL_STEPS = VAL_SIZE // GLOBAL_BATCH_SIZE

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(filepath=os.path.join(DIRS["checkpoints"], "dae_50k_best.keras"), save_best_only=True, monitor="val_loss"),
    keras.callbacks.CSVLogger(os.path.join(DIRS["logs"], "dae_training_log.csv"), append=True)
]

print(f"Starting Multi-GPU DAE Training across {strategy.num_replicas_in_sync} replicas...")
history = dae_model.fit(
    train_ds,
    steps_per_epoch=STEPS_PER_EPOCH,
    epochs=EPOCHS,
    validation_data=val_ds,
    validation_steps=VAL_STEPS,
    callbacks=callbacks
)

final_model_path = os.path.join(DIRS["models"], "DAE_50K_final.keras")
dae_model.save(final_model_path)
print(f"Saved final DAE model to {final_model_path}")


## 5. Quantitative Denoising Benchmark & Visual Verification

We evaluate the DAE on test samples by comparing:
1. **Input Noise Quality**: PSNR / SSIM of Noisy Input vs Clean Target.
2. **DAE Restored Quality**: PSNR / SSIM of DAE Output vs Clean Target.

In [ ]:
for noisy_batch, clean_batch in test_ds.take(1):
    sample_noisy = noisy_batch.numpy()
    sample_clean = clean_batch.numpy()
    break

denoised_batch = dae_model.predict(sample_noisy)

noisy_mse = np.mean(np.square(sample_noisy - sample_clean))
denoised_mse = np.mean(np.square(denoised_batch - sample_clean))

print(f"Mean Noisy Input MSE:   {noisy_mse:.6f}")
print(f"Mean DAE Denoised MSE: {denoised_mse:.6f}")
print(f"MSE Error Reduction:   {((noisy_mse - denoised_mse) / noisy_mse) * 100:.2f}%")

num_samples = 5
fig, axes = plt.subplots(3, num_samples, figsize=(15, 9))

for i in range(num_samples):
    axes[0, i].imshow(sample_noisy[i])
    axes[0, i].set_title("Noisy Input (sigma=0.15)")
    axes[0, i].axis("off")
    
    axes[1, i].imshow(denoised_batch[i])
    axes[1, i].set_title("DAE Denoised Output")
    axes[1, i].axis("off")
    
    axes[2, i].imshow(sample_clean[i])
    axes[2, i].set_title("Clean Target")
    axes[2, i].axis("off")

plt.suptitle("Denoising Autoencoder (DAE 50K) Performance Verification", fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "dae_performance_samples.png"), bbox_inches="tight")
plt.show()


## 6. Package Outputs for Download

In [ ]:
from IPython.display import FileLink

zip_output_path = "/kaggle/working/DAE_50K_Outputs"
shutil.make_archive(zip_output_path, "zip", BASE_DIR)

print(f"Successfully packaged DAE results to {zip_output_path}.zip!")
print("Download your trained DAE model and evaluation outputs using the link below:")
FileLink(r"DAE_50K_Outputs.zip")
